# V4 KOLF Data：可复现 QC、control-only HVG 与 VCC 覆盖验证

本 notebook 从原始 KOLF H5AD 开始，保留 V2 的 QC、原始 counts、标准化和绘图流程，并新增：

- 300 个 VCC 靶基因的逐项覆盖验证；
- `trainable` / `node_only` / `unavailable` 状态分类；
- 18,533 个 VCC 面板基因的覆盖与固定输出顺序；
- control-only HVG5000，避免 OOD 扰动条件参与 HVG 选择；
- 输入文件哈希、参数、软件版本和输出清单。

以后更换同格式 KOLF 数据文件时，**只需修改第一个代码单元中的 `INPUT_H5AD`**，其余代码和输出文件名不变。

In [ ]:
import scanpy as sc
import pandas as pd
import os
import sys
import json
import random
import hashlib
import platform
import datetime
import numpy as np
import gzip
import matplotlib.pyplot as plt
import re
import seaborn as sns
import scipy
from scipy.io import mmread
from scipy.sparse import csr_matrix, issparse
import tqdm.notebook as tq
import shutil

# =============================================================================
# V4 configuration：以后换同格式 KOLF 数据，只需修改 INPUT_H5AD。
# =============================================================================
INPUT_H5AD = r"E:\1Personal\New\科研起步\large_genetic_perturbation_20260909\02_kolf_ipsc\KOLF_ALPHA_first40000.h5ad"
PROJECT_DIR = r"E:\VSCODE\PrePR-CT\Cell-Type-Specific-Graphs-main"
VCC_DIR = r"E:\VSCODE\Virtual Cell Challenge\Original_data\vcc_2026_controls_final"
VCC_PERT_COUNTS = os.path.join(VCC_DIR, "pert_counts.csv")
VCC_GENE_NAMES = os.path.join(VCC_DIR, "gene_names.csv")

save_path_results = os.path.join(PROJECT_DIR, "Results")
save_path_data = os.path.join(PROJECT_DIR, "Data")
os.makedirs(save_path_results, exist_ok=True)
os.makedirs(save_path_data, exist_ok=True)

MIN_GENES = 200
MAX_PCT_MT = 5.0
N_HVG = 5000
RANDOM_SEED = 42
EXPECTED_VCC_TARGETS = 300
EXPECTED_VCC_GENES = 18533

FULL_OUTPUT_NAME = "KOLF_V4_full_QC.h5ad"
HVG_OUTPUT_NAME = "KOLF_V4_control_HVG5000.h5ad"
TARGET_COVERAGE_NAME = "V4_VCC_target_coverage.csv"
VCC_GENE_COVERAGE_NAME = "V4_VCC_gene_coverage.csv"
OUTPUT_ORDER_NAME = "V4_output_gene_order.csv"
CONDITION_COUNTS_NAME = "V4_KOLF_condition_counts.csv"
MANIFEST_NAME = "V4_data_manifest.json"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
sc.settings.seed = RANDOM_SEED

In [ ]:
# 只读取 INPUT_H5AD；不使用现成 QC_filtered 或 HVG5000 文件。
file_path = INPUT_H5AD
if not os.path.exists(file_path):
    raise FileNotFoundError(f"找不到输入文件：{file_path}")

adata_raw = sc.read_h5ad(file_path)
print(adata_raw)
print("X 类型：", type(adata_raw.X))

In [ ]:
# 对应 Kang 的 var 读取和检查。
var = adata_raw.var.copy()
var

In [ ]:
# 对应 Kang 的 obs 读取和检查。
obs = adata_raw.obs.copy()
obs

In [ ]:
# 对应 Kang 的表达矩阵读取；KOLF 矩阵已经是 cells × genes，不需要转置。
counts_mat_1 = csr_matrix(adata_raw.X[:])
counts_mat_1.shape

In [ ]:
# KOLF 只有一个表达矩阵，没有 Kang 的第二个 Matrix Market 文件。
# 保留此检查单元，明确不会合并其他数据集。
print("KOLF 输入矩阵数量：1")
print("细胞数 × 基因数：", counts_mat_1.shape)

In [ ]:
# 对应 Kang 的第一组 barcode／obs 对齐；H5AD 中 obs 已与矩阵行对齐。
obs_barcodes_1 = obs.copy()
if not obs_barcodes_1.index.equals(adata_raw.obs_names):
    raise ValueError("obs 与表达矩阵的细胞顺序不一致")
obs_barcodes_1

In [ ]:
# KOLF 没有第二组 barcode；保留单一输入检查。
required_obs = ["gRNA", "gene_target", "celltype", "perturbation_type", "channel"]
missing_obs = [c for c in required_obs if c not in obs_barcodes_1.columns]
if missing_obs:
    raise KeyError(f"KOLF obs 缺少必要字段：{missing_obs}")
print("必要字段检查通过：", required_obs)

In [ ]:
# Initialize an empty list to store AnnData objects
adata_list = []

# 使用 INPUT_H5AD 的原始 counts 和完整 KOLF metadata 创建 AnnData。
adata_1 = sc.AnnData(
    X=counts_mat_1,
    obs=obs_barcodes_1.copy(),
    var=var.copy(),
)

# V4 不额外添加 outlier 或 n_gRNA 过滤，严格保留用户指定的 QC 条件。
adata_1.var_names_make_unique()
adata_1.obs_names_make_unique()
adata_list.append(adata_1)
adata_1

In [ ]:
# Kang 在这里构建第二个数据集；KOLF 只有一个细胞类型和一个输入矩阵。
n_cell_types = adata_1.obs["celltype"].nunique(dropna=True)
if n_cell_types != 1:
    raise ValueError(f"本流程按单一细胞类型设计，但检测到 {n_cell_types} 种 celltype")
print("celltype：", adata_1.obs["celltype"].unique().tolist())

In [ ]:
# Kang 在这里 concat 两个数据集；KOLF 只有一个数据集，因此直接取出。
adata = adata_list[0]
adata.var_names_make_unique()
adata.obs_names_make_unique()

# 释放读取阶段的重复引用；adata 保留原始表达矩阵。
del adata_raw, counts_mat_1, adata_1, adata_list

In [ ]:
adata

In [ ]:
# 对应 Kang 的 adata.obs.stim.value_counts()。
adata.obs.gene_target.value_counts()

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('MT')
adata.var['ribo'] = adata.var_names.str.startswith(("RPS", "RPL"))

## Quality control: compute and visualise metrics

In [ ]:
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=['mt', 'ribo'],
    percent_top=None,
    log1p=False,
    inplace=True,
)

In [ ]:
# 所有 QC 图均实际运行，不跳过。
sc.pl.violin(
    adata, ['n_genes_by_counts'], jitter=0.4,
    groupby='celltype', rotation=45,
)
sc.pl.violin(
    adata, ['total_counts'], jitter=0.4,
    groupby='celltype', rotation=45,
)
sc.pl.violin(
    adata, ['pct_counts_mt'], jitter=0.4,
    groupby='celltype', rotation=45,
)
sc.pl.violin(
    adata, ['pct_counts_ribo'], jitter=0.4,
    groupby='celltype', rotation=45,
)
sns.scatterplot(
    data=adata.obs, x="total_counts", y="n_genes_by_counts", s=5
)
plt.show()

In [ ]:
sns.scatterplot(data=adata.obs, x="total_counts", y="pct_counts_mt", s=5)
plt.show()

In [ ]:
# 严格保留 Kang 的细胞／基因筛选结构，并套用 KOLF 阈值：
# n_genes_by_counts >= 200；pct_counts_mt <= 5；删除 mt／ribo 基因。
adata = adata[adata.obs.n_genes_by_counts >= MIN_GENES]
adata = adata[adata.obs.pct_counts_mt <= MAX_PCT_MT]
adata = adata[:, adata.var.mt == False]
adata = adata[:, adata.var.ribo == False]
adata = adata.copy()
adata


In [ ]:
# 对应 Kang 的三个字段重命名。
adata.obs.rename({"celltype": "cell_type"}, axis=1, inplace=True)
adata.obs.rename({"gene_target": "condition"}, axis=1, inplace=True)
adata.obs.rename({"channel": "replicate"}, axis=1, inplace=True)

# NTC 是对照；其他 condition 直接保留具体 CRISPRi 靶基因符号。
adata.obs["condition"] = adata.obs["condition"].astype(str).replace(
    {"NTC": "control", "non-targeting": "control"}
)

# 同时保留 KOLF 原始靶基因字段，便于后续追踪。
adata.obs["gene_target"] = adata.obs["condition"].replace(
    {"control": "NTC"}
)
adata.obs["perturbation_status"] = pd.Categorical(
    np.where(adata.obs["condition"] == "control", "control", "perturbed"),
    categories=["control", "perturbed"],
)

# 关键：完全按照 Kang，在 normalize_total 和 log1p 之前保存原始 counts。
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
# =============================================================================
# V4 VCC 靶基因验证：表达覆盖、control 特征可用性、直接扰动监督分开统计
# =============================================================================
def ordered_unique_genes(values):
    cleaned = pd.Series(values).dropna().astype(str).str.strip()
    cleaned = cleaned[(cleaned != '') & (cleaned.str.lower() != 'nan')]
    return list(dict.fromkeys(cleaned.tolist()))

for required_file in [VCC_PERT_COUNTS, VCC_GENE_NAMES]:
    if not os.path.exists(required_file):
        raise FileNotFoundError(f'找不到 VCC 文件：{required_file}')

pert_df = pd.read_csv(VCC_PERT_COUNTS)
target_candidates = [
    'target_gene', 'target', 'gene_name', 'gene',
    'perturbation', 'pert_name', 'gene_target'
]
lower_to_original = {str(c).lower(): c for c in pert_df.columns}
target_col = next(
    (lower_to_original[name] for name in target_candidates if name in lower_to_original),
    None,
)
if target_col is None:
    raise KeyError(
        '无法识别 pert_counts.csv 的靶基因列。'
        f'实际列为：{pert_df.columns.tolist()}；请检查官方文件格式。'
    )
vcc_target_genes = ordered_unique_genes(pert_df[target_col])

vcc_gene_df_raw = pd.read_csv(VCC_GENE_NAMES)
gene_col = 'gene_name' if 'gene_name' in vcc_gene_df_raw.columns else vcc_gene_df_raw.columns[0]
vcc_panel_genes = ordered_unique_genes(vcc_gene_df_raw[gene_col])

if len(vcc_target_genes) != EXPECTED_VCC_TARGETS:
    print(f'警告：预期 {EXPECTED_VCC_TARGETS} 个 VCC 靶基因，实际读取 {len(vcc_target_genes)} 个。')
if len(vcc_panel_genes) != EXPECTED_VCC_GENES:
    print(f'警告：预期 {EXPECTED_VCC_GENES} 个 VCC 面板基因，实际读取 {len(vcc_panel_genes)} 个。')

# control 中实际检出情况：基因符号存在不等于在 control 中有表达。
ctrl_all = adata[adata.obs.condition == 'control']
if ctrl_all.n_obs == 0:
    raise ValueError('没有 control 细胞。')
X_ctrl = ctrl_all.X
if issparse(X_ctrl):
    control_detected_counts = np.asarray(X_ctrl.getnnz(axis=0)).ravel()
    control_mean_log1p = np.asarray(X_ctrl.mean(axis=0)).ravel()
else:
    X_ctrl_dense = np.asarray(X_ctrl)
    control_detected_counts = (X_ctrl_dense > 0).sum(axis=0)
    control_mean_log1p = X_ctrl_dense.mean(axis=0)

adata.var['control_n_cells_detected'] = control_detected_counts.astype(int)
adata.var['control_detection_rate'] = control_detected_counts / ctrl_all.n_obs
adata.var['control_mean_log1p'] = control_mean_log1p

gene_to_var_index = {g: i for i, g in enumerate(adata.var_names.astype(str))}
kolf_gene_set = set(gene_to_var_index)
condition_counts = adata.obs['condition'].value_counts()
kolf_condition_targets = set(
    adata.obs.loc[adata.obs.condition != 'control', 'condition'].astype(str)
)

coverage_rows = []
for vcc_index, gene in enumerate(vcc_target_genes):
    in_expression = gene in kolf_gene_set
    in_condition = gene in kolf_condition_targets
    var_index = gene_to_var_index.get(gene, -1)
    detected = int(adata.var.iloc[var_index]['control_n_cells_detected']) if in_expression else 0
    detection_rate = float(adata.var.iloc[var_index]['control_detection_rate']) if in_expression else 0.0
    mean_log1p = float(adata.var.iloc[var_index]['control_mean_log1p']) if in_expression else 0.0
    if in_expression and in_condition:
        status = 'trainable'
    elif in_expression:
        status = 'node_only'
    else:
        status = 'unavailable'
    coverage_rows.append({
        'vcc_index': vcc_index,
        'target_gene': gene,
        'in_kolf_expression': in_expression,
        'in_kolf_condition': in_condition,
        'node_feature_available': in_expression,
        'control_feature_available': bool(in_expression and detected > 0),
        'direct_training_available': bool(in_expression and in_condition),
        'status': status,
        'n_kolf_perturbed_cells': int(condition_counts.get(gene, 0)),
        'n_control_cells_detected': detected,
        'control_detection_rate': detection_rate,
        'control_mean_log1p': mean_log1p,
    })

target_coverage_df = pd.DataFrame(coverage_rows)
print('VCC targets total:', len(vcc_target_genes))
print('In KOLF expression:', int(target_coverage_df.in_kolf_expression.sum()))
print('In KOLF conditions:', int(target_coverage_df.in_kolf_condition.sum()))
print('Directly trainable:', int(target_coverage_df.direct_training_available.sum()))
print('Node-only:', int((target_coverage_df.status == 'node_only').sum()))
print('Unavailable:', int((target_coverage_df.status == 'unavailable').sum()))
print('Present but without control detection:', int((target_coverage_df.in_kolf_expression & ~target_coverage_df.control_feature_available).sum()))
print('Unavailable targets:', target_coverage_df.loc[
    target_coverage_df.status == 'unavailable', 'target_gene'
].tolist())

if not target_coverage_df.in_kolf_expression.any():
    raise ValueError('VCC 靶基因与当前 KOLF 表达矩阵没有重叠，请检查数据或基因符号。')

adata.var['in_vcc_target'] = adata.var_names.isin(set(vcc_target_genes))
status_lookup = dict(zip(target_coverage_df.target_gene, target_coverage_df.status))
adata.var['vcc_target_status'] = [
    status_lookup.get(g, '') for g in adata.var_names.astype(str)
]

condition_counts_df = pd.DataFrame({
    'condition': condition_counts.index.astype(str),
    'n_cells': condition_counts.values.astype(int),
})
condition_counts_df['in_vcc_targets'] = condition_counts_df.condition.isin(set(vcc_target_genes))
condition_counts_df.to_csv(
    os.path.join(save_path_data, CONDITION_COUNTS_NAME), index=False
)

target_coverage_df.head()

In [ ]:
# V4：HVG 只由 control 细胞选择，避免 OOD perturbed 条件参与基因筛选。
ctrl_for_hvg = adata[adata.obs.condition == 'control'].copy()
if ctrl_for_hvg.n_obs < 2:
    raise ValueError('control 细胞不足，无法选择 HVG。')

n_hvg_actual = min(N_HVG, ctrl_for_hvg.n_vars)
sc.pp.highly_variable_genes(
    ctrl_for_hvg, subset=False, n_top_genes=n_hvg_actual
)
hvg_genes = ctrl_for_hvg.var_names[
    ctrl_for_hvg.var['highly_variable'].astype(bool)
].tolist()
hvg_gene_set = set(hvg_genes)
del ctrl_for_hvg

adata.var['control_hvg'] = adata.var_names.isin(hvg_gene_set)
other_genes = [g for g in adata.var_names if g not in hvg_gene_set]

# full_QC 保留全部 QC 基因；control HVG 固定排在前面，便于复现和审计。
adata_full = adata[:, hvg_genes + other_genes].copy()
adata_hvg = adata_full[:, hvg_genes].copy()

print('Control-selected HVG genes:', adata_hvg.n_vars)
print('Full-QC genes:', adata_full.n_vars)
print(
    'Full output first genes equal control HVG:',
    adata_full.var_names[:len(hvg_genes)].tolist() == adata_hvg.var_names.tolist()
)
adata_hvg

In [ ]:
# =============================================================================
# V4 VCC 全基因面板覆盖与固定输出顺序
# =============================================================================
full_gene_to_index = {g: i for i, g in enumerate(adata_full.var_names.astype(str))}
output_gene_records = []
vcc_gene_coverage_rows = []

for official_index, gene in enumerate(vcc_panel_genes):
    in_expression = gene in full_gene_to_index
    full_index = full_gene_to_index.get(gene, -1)
    detected = int(adata_full.var.iloc[full_index]['control_n_cells_detected']) if in_expression else 0
    detection_rate = float(adata_full.var.iloc[full_index]['control_detection_rate']) if in_expression else 0.0
    vcc_gene_coverage_rows.append({
        'official_index': official_index,
        'gene_name': gene,
        'in_kolf_expression': in_expression,
        'full_qc_index': full_index,
        'n_control_cells_detected': detected,
        'control_detection_rate': detection_rate,
    })
    if in_expression:
        output_gene_records.append({
            'output_index': len(output_gene_records),
            'official_index': official_index,
            'gene_name': gene,
            'full_qc_index': full_index,
        })

vcc_gene_coverage_df = pd.DataFrame(vcc_gene_coverage_rows)
output_order_df = pd.DataFrame(output_gene_records)
output_genes = output_order_df.gene_name.tolist()
missing_vcc_genes = vcc_gene_coverage_df.loc[
    ~vcc_gene_coverage_df.in_kolf_expression, 'gene_name'
].tolist()

print('KOLF full-QC genes:', adata_full.n_vars)
print('VCC genes:', len(vcc_panel_genes))
print('VCC genes covered by KOLF:', len(output_genes), '/', len(vcc_panel_genes))
print('VCC genes missing in KOLF:', len(missing_vcc_genes))
print('First 30 missing VCC genes:', missing_vcc_genes[:30])
if not output_genes:
    raise ValueError('VCC 基因面板与 KOLF 没有重叠。')

# 给 target coverage 补上最终全基因位置和官方输出位置。
output_index_lookup = dict(zip(output_order_df.gene_name, output_order_df.output_index))
target_coverage_df['full_qc_index'] = [
    full_gene_to_index.get(g, -1) for g in target_coverage_df.target_gene
]
target_coverage_df['vcc_output_index'] = [
    output_index_lookup.get(g, -1) for g in target_coverage_df.target_gene
]

vcc_gene_coverage_df.to_csv(
    os.path.join(save_path_data, VCC_GENE_COVERAGE_NAME), index=False
)
output_order_df.to_csv(
    os.path.join(save_path_data, OUTPUT_ORDER_NAME), index=False
)
target_coverage_df.to_csv(
    os.path.join(save_path_data, TARGET_COVERAGE_NAME), index=False
)

adata_full.var['in_vcc_panel'] = adata_full.var_names.isin(set(vcc_panel_genes))
adata_full.var['vcc_output_index'] = [
    output_index_lookup.get(g, -1) for g in adata_full.var_names.astype(str)
]
adata_hvg.var['in_vcc_panel'] = adata_hvg.var_names.isin(set(vcc_panel_genes))
adata_hvg.var['vcc_output_index'] = [
    output_index_lookup.get(g, -1) for g in adata_hvg.var_names.astype(str)
]

output_order_df.head()

In [ ]:
sc.pp.pca(adata_hvg, random_state=RANDOM_SEED)
sc.pp.neighbors(adata_hvg, random_state=RANDOM_SEED)
sc.tl.umap(adata_hvg, random_state=RANDOM_SEED)

In [ ]:
adata_hvg.obs['replicate'] = adata_hvg.obs['replicate'].astype(str)
adata_hvg

In [ ]:
# 保留 Kang 的双面板 UMAP，并实际显示和保存。
# 第一面板显示 control／perturbed，避免数百个靶基因超出调色板类别上限。
fig, axes = plt.subplots(nrows=2, figsize=(8, 8))

sc.pl.umap(
    adata=adata_hvg,
    color=["perturbation_status"],
    legend_fontsize=18,
    ax=axes[0],
    title="Condition",
    show=False,
    palette=sns.color_palette("hls", 2),
    frameon=False,
)

sc.pl.umap(
    adata=adata_hvg,
    color=["cell_type"],
    legend_fontsize=18,
    ax=axes[1],
    title="Cell type",
    show=False,
    palette=sns.color_palette("hls", 1),
    frameon=False,
)

plt.tight_layout()
plt.savefig(os.path.join(save_path_results, "KOLF_V4_UMAP.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# 对应 Kang 的 counts／X 最大值检查；稀疏矩阵不做 dense 转换。
def matrix_max(matrix):
    return matrix.max() if issparse(matrix) else np.max(matrix)

print("layers['counts'] max:", matrix_max(adata_hvg.layers['counts']))
print("X max:", matrix_max(adata_hvg.X))

In [ ]:
# 对应 Kang 的 stimulated 细胞统计；KOLF 为所有非 NTC 的 CRISPRi 扰动。
adata_hvg[adata_hvg.obs.condition != 'control'].copy().obs.cell_type.value_counts()

In [ ]:
adata_hvg[adata_hvg.obs.condition == 'control'].copy().obs.cell_type.value_counts()

In [ ]:
adata_hvg.obs['cov_cond'] = (
    adata_hvg.obs.cell_type.astype(str)
    + '_'
    + adata_hvg.obs.condition.astype(str)
)
adata_full.obs['cov_cond'] = adata_hvg.obs['cov_cond']

In [ ]:
# V4 固定输出：全 QC 基因 + control-only HVG5000。
for output_object, output_name in [
    (adata_full, FULL_OUTPUT_NAME),
    (adata_hvg, HVG_OUTPUT_NAME),
]:
    if "counts" not in output_object.layers:
        raise KeyError(f"保存失败：{output_name} 缺少 layers['counts']")
    if output_object.layers["counts"].shape != output_object.X.shape:
        raise ValueError(f"counts 与 X 的形状不一致：{output_name}")
    output_file = os.path.join(save_path_data, output_name)
    output_object.write(output_file)
    print("已保存：", output_file, output_object.shape)

if not adata_full.obs_names.equals(adata_hvg.obs_names):
    raise ValueError('full_QC 与 control_HVG 文件的细胞顺序不一致。')

In [ ]:
# =============================================================================
# V4 reproducibility manifest
# =============================================================================
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

input_stat = os.stat(file_path)
manifest = {
    'version': 'V4',
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'random_seed': RANDOM_SEED,
    'input': {
        'h5ad': file_path,
        'size_bytes': input_stat.st_size,
        'mtime_ns': input_stat.st_mtime_ns,
        'sha256': sha256_file(file_path),
    },
    'qc': {
        'min_genes_by_counts': MIN_GENES,
        'max_pct_counts_mt': MAX_PCT_MT,
        'remove_mt_genes': True,
        'remove_ribo_genes': True,
        'require_single_cell_type': True,
        'additional_outlier_or_n_gRNA_filters': False,
    },
    'preprocessing': {
        'counts_layer_saved_before_normalization': True,
        'normalization': 'sc.pp.normalize_total followed by sc.pp.log1p',
        'hvg_selection': 'control cells only',
        'n_hvg_requested': N_HVG,
        'n_hvg_actual': len(hvg_genes),
    },
    'dataset': {
        'n_cells': int(adata_full.n_obs),
        'n_full_qc_genes': int(adata_full.n_vars),
        'n_control_cells': int((adata_full.obs.condition == 'control').sum()),
        'n_perturbed_cells': int((adata_full.obs.condition != 'control').sum()),
        'n_kolf_condition_targets': len(kolf_condition_targets),
    },
    'vcc_targets': {
        'n_total': len(vcc_target_genes),
        'n_in_kolf_expression': int(target_coverage_df.in_kolf_expression.sum()),
        'n_in_kolf_condition': int(target_coverage_df.in_kolf_condition.sum()),
        'n_directly_trainable': int(target_coverage_df.direct_training_available.sum()),
        'n_node_only': int((target_coverage_df.status == 'node_only').sum()),
        'n_unavailable': int((target_coverage_df.status == 'unavailable').sum()),
        'n_present_without_control_detection': int((target_coverage_df.in_kolf_expression & ~target_coverage_df.control_feature_available).sum()),
    },
    'vcc_gene_panel': {
        'n_total': len(vcc_panel_genes),
        'n_covered_by_kolf': len(output_genes),
        'n_missing_in_kolf': len(missing_vcc_genes),
    },
    'outputs': {
        'full_qc_h5ad': os.path.join(save_path_data, FULL_OUTPUT_NAME),
        'control_hvg_h5ad': os.path.join(save_path_data, HVG_OUTPUT_NAME),
        'target_coverage_csv': os.path.join(save_path_data, TARGET_COVERAGE_NAME),
        'vcc_gene_coverage_csv': os.path.join(save_path_data, VCC_GENE_COVERAGE_NAME),
        'output_gene_order_csv': os.path.join(save_path_data, OUTPUT_ORDER_NAME),
        'condition_counts_csv': os.path.join(save_path_data, CONDITION_COUNTS_NAME),
        'umap_pdf': os.path.join(save_path_results, 'KOLF_V4_UMAP.pdf'),
    },
    'software': {
        'python': platform.python_version(),
        'scanpy': sc.__version__,
        'anndata': __import__('anndata').__version__,
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scipy': scipy.__version__,
    },
}
manifest_path = os.path.join(save_path_data, MANIFEST_NAME)
with open(manifest_path, 'w', encoding='utf-8') as handle:
    json.dump(manifest, handle, ensure_ascii=False, indent=2)
print('V4 manifest saved to:', manifest_path)
print(json.dumps({
    'vcc_targets': manifest['vcc_targets'],
    'vcc_gene_panel': manifest['vcc_gene_panel'],
}, ensure_ascii=False, indent=2))

In [ ]:
adata_hvg

In [ ]:
ctrl = adata_hvg[adata_hvg.obs.condition == 'control'].copy()
print(ctrl.obs.replicate.value_counts())
pd.crosstab(adata_hvg.obs['cell_type'], adata_hvg.obs['replicate'])